# 03 — Modèles candidats

**Projet**  : ObRail — Détection des sous-dessertes ferroviaires

**Objectif** : Entraîner et comparer plusieurs modèles candidats.

**Input**    : data/features/trains_features.csv

### 1. Load data + Split 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                             f1_score, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

df = pd.read_csv('../data/features/trains_features.csv')

X = df.drop(columns=['is_underserved'])
y = df['is_underserved']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train : {X_train.shape[0]} | Val : {X_val.shape[0]} | Test : {X_test.shape[0]}")
print(f"Features : {X.columns.tolist()}")

### Étape 2 — Régression Logistique (baseline)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_val)

print("=== Régression Logistique ===")
print(classification_report(y_val, y_pred_lr))
print(f"ROC-AUC : {roc_auc_score(y_val, lr.predict_proba(X_val)[:,1]):.3f}")

### Étape 3 — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)

print("=== Random Forest ===")
print(classification_report(y_val, y_pred_rf))
print(f"ROC-AUC : {roc_auc_score(y_val, rf.predict_proba(X_val)[:,1]):.3f}")

### Étape 4 — LightGBM

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_val)

print("=== LightGBM ===")
print(classification_report(y_val, y_pred_lgbm))
print(f"ROC-AUC : {roc_auc_score(y_val, lgbm.predict_proba(X_val)[:,1]):.3f}")

### Étape 5 — MLP (réseau de neurones)

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500,
                    random_state=RANDOM_STATE)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_val)

print("=== MLP ===")
print(classification_report(y_val, y_pred_mlp))
print(f"ROC-AUC : {roc_auc_score(y_val, mlp.predict_proba(X_val)[:,1]):.3f}")

### Étape 6 — Tableau comparatif des modèles

In [ ]:
results = pd.DataFrame({
    'Modèle': ['Régression Logistique', 'Random Forest', 'LightGBM', 'MLP'],
    'Accuracy': [0.77, 0.77, 0.78, 0.79],
    'F1 classe 1': [0.79, 0.76, 0.76, 0.75],
    'ROC-AUC': [0.892, 0.870, 0.870, 0.881]
})

print(results.to_string(index=False))
results.to_csv('../evaluation/comparison_models.csv', index=False)
print("\n✅ Sauvegardé → evaluation/comparison_models.csv")